In [16]:
import httpx
import requests
from urllib.parse import urlencode, quote
import logging

In [17]:
logger = logging.getLogger(__name__)

In [3]:
response = requests.get("https://arxiv.org/abs/2305.16216")

In [18]:
def get_arxiv_url() -> str:
    params = {
            "search_query": "cat:cs.AI",
            "start": 0,
            "max_results": 1,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
    url = f"https://export.arxiv.org/api/query?{urlencode(params, quote_via=quote, safe=":+[]")}"
    return url


In [23]:
async def get_papers_from_arxiv() -> str:
    try:
        url = get_arxiv_url()

        async with httpx.AsyncClient(timeout=60) as httpclient:
            response = await httpclient.get(url)
            response.raise_for_status()
            return response.text

    except httpx.TimeoutException:
        logger.exception("Timeout while calling arXiv API")
        raise

    except httpx.HTTPStatusError:
        logger.exception("arXiv API returned an error response")
        raise

    except Exception:
        logger.exception("Unexpected error occurred while calling arXiv API")
        raise

In [25]:
result = await get_papers_from_arxiv()
print(result)

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/HQttLwSI9ar2vqfmj4lJ7u3JUas</id>
  <title>arXiv Query: search_query=cat:cs.AI&amp;id_list=&amp;start=0&amp;max_results=1</title>
  <updated>2026-06-10T15:13:40Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.AI&amp;start=0&amp;max_results=1&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>1</opensearch:itemsPerPage>
  <opensearch:totalResults>183838</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2606.11189v1</id>
    <title>A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design</title>
    <updated>2026-06-09T17:59:54Z</updated>
    <link href="https://arxiv.org/abs/2606.11189v1" rel="alternate" type="text/html"/>
    <link href="https://arxiv.

In [32]:
from pydantic import BaseModel, HttpUrl
from typing import List
from datetime import datetime

class ArxivPaperAPI(BaseModel):
    title: str
    authors: List[str]
    arxiv_id: str
    summary: str
    categories: List[str]
    published_at: datetime
    pdf_url: HttpUrl

In [33]:
import logging
import xml.etree.ElementTree as ET
from datetime import datetime
from typing import List, Optional

logger = logging.getLogger(__name__)


class ArxivXmlParser:
    """
    Parse arXiv Atom XML responses into ArxivPaper objects.
    """

    def __init__(self):
        self._namespaces : dict = {
        "atom": "http://www.w3.org/2005/Atom",
        "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
        "arxiv": "http://arxiv.org/schemas/atom",
    }

    def parse(self, xml_data: str) -> List[ArxivPaperAPI]:
        """
        Parse an arXiv XML response into a list of papers.
        """
        try:
            root = ET.fromstring(xml_data)
            entries = root.findall("atom:entry", self._namespaces)

            papers = []

            for entry in entries:
                paper = self._parse_entry(entry)

                if paper:
                    papers.append(paper)

            return papers

        except ET.ParseError as e:
            logger.error(f"Malformed XML from arXiv: {e}")
            raise

        except Exception as e:
            logger.exception("Unexpected error parsing arXiv response")
            raise
        
    def _parse_entry(self, entry: ET.Element) -> Optional[ArxivPaperAPI]:
        try:
            arxiv_id = self._get_id(entry)
            title = self._get_text(entry, "atom:title")
            summary = self._get_text(entry, "atom:summary")
            categories = self._get_categories(entry)
            published_at = self._get_published_at(entry)
            authors = self._get_authors(entry)
            pdf_url = self._get_pdf_url(entry)
            
            if not arxiv_id:
                logger.warning("Skipping entry with missing arXiv ID")
                return None

            return ArxivPaperAPI(
                title= title,
                authors=authors,
                arxiv_id=arxiv_id,
                summary=summary,
                categories=categories,
                published_at=published_at,
                pdf_url=pdf_url,
            )

        except Exception:
            logger.exception("Failed to parse arXiv entry")
            return None

    def _get_text(self, element: ET.Element, path: str) -> str:
        elem = element.find(path, self._namespaces)

        if elem is None or elem.text is None:
            return ""

        text = elem.text.strip()
        text = " ".join(text.split())

        return text

    def _get_id(self, entry: ET.Element) -> Optional[str]:
        id_elem = entry.find("atom:id", self._namespaces)

        if id_elem is None or id_elem.text is None:
            return None

        return id_elem.text.strip().split("/")[-1]

    def _get_authors(self, entry: ET.Element) -> List[str]:
        authors = []

        for author in entry.findall("atom:author", self._namespaces):
            name = self._get_text(author, "atom:name")

            if name:
                authors.append(name)

        return authors

    def _get_categories(self,entry: ET.Element) -> List[str]:
        categories = []

        for category in entry.findall("atom:category", self._namespaces):
            term = category.get("term")

            if term:
                categories.append(term)

        return categories

    def _get_published_at(self, entry: ET.Element) -> datetime:
        published = self._get_text(
            entry,
            "atom:published",
        )

        return datetime.fromisoformat(published.replace("Z", "+00:00"))

    def _get_pdf_url(self, entry: ET.Element) -> str:
        for link in entry.findall("atom:link", self._namespaces):
            if link.get("type") == "application/pdf":
                url = link.get("href", "")

                if url.startswith("http://arxiv.org/"):
                    url = url.replace(
                        "http://arxiv.org/",
                        "https://arxiv.org/",
                    )

                return url

        return ""

In [ ]:
def parse_xml_to_arxivpaperapi(xml_data:str) -> List[ArxivPaperAPI]:
    xml_parser = ArxivXmlParser()
    response = xml_parser.parse(xml_data=xml_data)
    
    return response

result = await get_papers_from_arxiv()
list_arxiv_paper = parse_xml_to_arxivpaperapi(result)


[ArxivPaperAPI(title='A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design', authors=['Tong Xie', 'Yuanhao Ban', 'Yunqi Hong', 'Sohyun An', 'Yihang Chen', 'Cho-Jui Hsieh'], arxiv_id='2606.11189v1', summary='Supervised fine-tuning (SFT) typically maximizes the likelihood of every token in a demonstrated trajectory. However, an observed token can be non-unique, noisy, or misaligned with the model prior. Strictly fitting toward this one-hot target may be suboptimal, especially when the pretrained model encodes a rich knowledge prior. In this work, we reinterpret SFT as target distribution design: instead of studying only the loss objective, we analyze the token-level target that the loss drives the model to match. We introduce the Q-target framework, which decomposes SFT supervision into two explicit choices: (1) how strongly to rely on the observed token, and (2) how to allocate the remaining probability mass over alternatives. This perspective unifies many existi

In [36]:
for paper in list_arxiv_paper:
    print(f"{paper.arxiv_id}\n{paper.title}\n{paper.authors}\n{paper.categories}\n{paper.summary}\n{paper.published_at}\n{paper.pdf_url}")

2606.11189v1
A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design
['Tong Xie', 'Yuanhao Ban', 'Yunqi Hong', 'Sohyun An', 'Yihang Chen', 'Cho-Jui Hsieh']
['cs.LG', 'cs.AI', 'cs.CL']
Supervised fine-tuning (SFT) typically maximizes the likelihood of every token in a demonstrated trajectory. However, an observed token can be non-unique, noisy, or misaligned with the model prior. Strictly fitting toward this one-hot target may be suboptimal, especially when the pretrained model encodes a rich knowledge prior. In this work, we reinterpret SFT as target distribution design: instead of studying only the loss objective, we analyze the token-level target that the loss drives the model to match. We introduce the Q-target framework, which decomposes SFT supervision into two explicit choices: (1) how strongly to rely on the observed token, and (2) how to allocate the remaining probability mass over alternatives. This perspective unifies many existing SFT variants as implici

After converting ArxivAPIPaper 
it should now be having pdf_url to get the paper downloaded. Again call get endpoint.
After downloading, docling helps to parse the paper. 
once it is parsed then store in database.